# 03 — Context Engineering and Cache-Augmented Generation

**First Finance - Arnaud Demes**  
**Session:** Day 1, 10:45–11:30 · 15 minutes concepts + 30 minutes notebook

Build a document-grounded financial answer **without retrieval**, then make the
failure boundary visible before Lesson 4 introduces RAG.

## Learning objectives

By the end of this notebook, you can:

1. distinguish context engineering from prompt wording;
2. place a complete local document before the changing question;
3. estimate the full input budget, including instructions and reserved output;
4. explain what prompt caching can and cannot optimize;
5. recognize when direct context should give way to retrieval.

> **Deliverable:** one grounded NVIDIA answer and one explicit CAG/RAG decision.

## Before you start

- Run the notebook from the repository root environment.
- Default live provider: local Ollama (`qwen3:8b`).
- Hosted option: set `FINAI_MODEL_PROVIDER=openai` and `OPENAI_API_KEY`.
- Automated tests set `FINAI_LIVE_MODE=0` and use a deterministic recorded model.
- The baseline path does not download a filing or require a paid API.

The document below is a **curated teaching extract** from NVIDIA's fiscal 2026
Form 10-K. It is complete for this bounded exercise, but it is not the full filing.

## Where this fits

```text
Lesson 1             Lesson 2                 Lesson 3                    Lesson 4
model boundary  →  typed financial output  →  complete-document CAG  →  retrieval + RAG
```

Lesson 2 controlled the **shape** of the answer. Lesson 3 controls the **information
available to the model**. Lesson 4 will select only the passages needed for each question.

| Concept | What it controls | What it does not prove |
|---|---|---|
| Context | Information supplied for this call | Long-term recall |
| Cache | Reuse of an exact stable prefix | Grounding or correctness |
| Memory | State retained across interactions | That a source supports a claim |
| Grounding | Whether claims are supported by evidence | That the whole document should be sent |
| RAG | Which evidence units enter context | That retrieval or generation is correct |

The notebook returns a typed `ContextDecision` so the CAG/RAG route and its budget
reason remain inspectable application state.

## Set up the lab

All charts are generated from values computed in the notebook. They are part of the
engineering evidence, not decorative illustrations.

In [ ]:
from __future__ import annotations

import os
from time import perf_counter
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from finai_academy.context import (
    ContextBudget,
    build_full_context_prompt,
    decide_context_route,
    estimate_tokens,
    should_use_full_context,
)
from finai_academy.lesson_support import RecordedChatModel, evaluate_grounding
from finai_academy.providers import ModelRun, create_chat_model, provider_summary
from finai_academy.settings import Settings

NAVY = "#102A43"
TEAL = "#12A594"
ORANGE = "#F0A23B"
RED = "#D95D5D"
SLATE = "#627D98"
LIGHT = "#D9E2EC"

plt.rcParams.update({
    "figure.figsize": (10, 4.8),
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "font.size": 10,
})

## 1. Load a real financial document pack

The source facts come from NVIDIA's fiscal 2026 Form 10-K filed with the SEC.
We keep provenance beside the text so a later ingestion pipeline can preserve it.

**Official source:** [NVIDIA FY2026 Form 10-K](https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm)

In [ ]:
SOURCE_URL = (
    "https://www.sec.gov/Archives/edgar/data/1045810/"
    "000104581026000021/nvda-20260125.htm"
)

document_sections = [
    {
        "id": "F1",
        "section": "Results of operations",
        "text": (
            "For fiscal 2026, NVIDIA reported revenue of $215.938 billion, "
            "compared with $130.497 billion in fiscal 2025 and $60.922 billion "
            "in fiscal 2024. The filing describes fiscal 2026 revenue as up 65% "
            "from the prior year."
        ),
    },
    {
        "id": "F2",
        "section": "Revenue by end market",
        "text": (
            "Data Center revenue was $193.737 billion in fiscal 2026, versus "
            "$115.186 billion in fiscal 2025. The filing reports 68% year-on-year "
            "growth and attributes growth to accelerated computing and AI."
        ),
    },
    {
        "id": "F3",
        "section": "Gaming",
        "text": (
            "Gaming revenue was $16.042 billion in fiscal 2026, compared with "
            "$11.350 billion in fiscal 2025. NVIDIA reports 41% growth, driven by "
            "Blackwell demand."
        ),
    },
    {
        "id": "F4",
        "section": "Gross profit",
        "text": (
            "Gross profit was $153.463 billion on revenue of $215.938 billion. "
            "The filing says gross margin decreased as the business transitioned "
            "from Hopper HGX systems to Blackwell full-scale data-center solutions."
        ),
    },
    {
        "id": "F5",
        "section": "Inventory charge",
        "text": (
            "The gross-margin decrease was also affected by a $4.5 billion charge "
            "associated with H20 excess inventory and purchase obligations."
        ),
    },
    {
        "id": "F6",
        "section": "Geographic revenue",
        "text": (
            "Revenue based on customer headquarters was $149.617 billion in the "
            "United States, $42.345 billion in Taiwan and $19.677 billion in China, "
            "including Hong Kong."
        ),
    },
    {
        "id": "F7",
        "section": "Research and development",
        "text": (
            "Research and development expense was $18.497 billion in fiscal 2026, "
            "compared with $12.914 billion in fiscal 2025."
        ),
    },
    {
        "id": "F8",
        "section": "Forward-looking constraint",
        "text": (
            "Management expected supply constraints to remain a headwind to Gaming "
            "in the first quarter of fiscal 2027 and beyond. This is a management "
            "expectation, not a reported fiscal 2026 result."
        ),
    },
]

source_document = "\n\n".join(
    f"[{item['id']}] {item['section']}\n{item['text']}" for item in document_sections
)

section_table = pd.DataFrame(document_sections)
section_table["estimated_tokens"] = section_table["text"].map(estimate_tokens)
section_table[["id", "section", "estimated_tokens"]]

### Visual 1 — Where the source budget goes

Before calling a model, inspect which sections dominate the source. This becomes
essential when a report contains long notes, tables and appendices.

In [ ]:
plot_table = section_table.sort_values("estimated_tokens")
colors = [TEAL if fact_id in {"F1", "F2"} else LIGHT for fact_id in plot_table["id"]]

fig, ax = plt.subplots()
bars = ax.barh(
    plot_table["section"],
    plot_table["estimated_tokens"],
    color=colors,
    edgecolor=NAVY,
    linewidth=0.6,
)
ax.bar_label(bars, padding=4, fmt="%.0f")
ax.set_title("Estimated tokens by filing section")
ax.set_xlabel("Estimated tokens · four-character approximation")
ax.set_ylabel("Source section")
ax.spines[["top", "right"]].set_visible(False)
ax.text(
    0.99,
    0.04,
    "Teal = sections needed for the first question",
    transform=ax.transAxes,
    ha="right",
    color=SLATE,
)
plt.tight_layout()
plt.show()

## 2. Budget the complete prompt

A context window is shared by four components:

1. system instructions;
2. the source document;
3. the changing question;
4. space reserved for the answer.

The estimator below is deliberately approximate and provider-neutral. Production
systems should use the tokenizer published for the selected model.

In [ ]:
SYSTEM_INSTRUCTIONS = (
    "Use only the supplied source. Cite fact identifiers after factual claims. "
    "Separate reported facts from interpretations and state any limitation."
)
QUESTION_A = (
    "Return three concise bullets labelled Growth, Concentration and Limitation. "
    "Start Growth with 'NVIDIA fiscal 2026'. Use only F1 and F2, cite them in "
    "square brackets [F1] and [F2], and round monetary values to one decimal place. "
    "Do not calculate ratios, cite any other fact, or provide a recommendation. "
    "End Limitation with exactly: 'The supplied evidence does not establish "
    "valuation or a price target.'"
)

budget = ContextBudget(max_input_tokens=8_192, reserved_output_tokens=1_200)
components = {
    "Instructions": estimate_tokens(SYSTEM_INSTRUCTIONS),
    "Complete document": estimate_tokens(source_document),
    "Question": estimate_tokens(QUESTION_A),
    "Reserved output": budget.reserved_output_tokens,
}

base_decision = decide_context_route(
    document_tokens=components["Complete document"],
    system_prompt_tokens=components["Instructions"],
    question_tokens=components["Question"],
    budget=budget,
)
base_fits = base_decision.route == "cag"

print("Available input tokens:", budget.available_input_tokens)
print("Estimated prompt components:", components)
print("ContextDecision:", base_decision)
print("Full document decision:", "CAG fits" if base_fits else "RAG required")

### Visual 2 — The context window is an allocation problem

The reserved output is not optional: a prompt that consumes the entire window leaves
the model nowhere to produce the requested answer.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.8))
left = 0
palette = [NAVY, TEAL, ORANGE, SLATE]
for (label, value), color in zip(components.items(), palette, strict=True):
    ax.barh([0], [value], left=left, color=color, label=f"{label}: {value:,}")
    if value >= 120:
        ax.text(left + value / 2, 0, f"{value:,}", ha="center", va="center", color="white")
    left += value

ax.axvline(budget.max_input_tokens, color=RED, linewidth=2, label="Model context limit")
ax.set_xlim(0, budget.max_input_tokens * 1.03)
ax.set_yticks([])
ax.set_xlabel("Estimated tokens")
ax.set_title("Complete prompt allocation inside an 8,192-token window")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.25), frameon=False)
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Build a cache-friendly CAG prompt

Cache-Augmented Generation (CAG) reuses a stable long prefix across related
questions. The key design rule is simple:

```text
stable instructions → stable source document → changing question
```

Prompt caching can reduce processing cost or latency when a provider finds an exact
eligible prefix. It does **not** improve factual accuracy, expand the context window or
guarantee the same answer. Provider behavior must be measured from returned usage data.

In [ ]:
QUESTION_B = (
    "Using F1 and F2, explain in two sentences why fiscal 2026 growth was "
    "concentrated. Cite both identifiers and state one limitation."
)

prompt_a = build_full_context_prompt(
    document_text=source_document,
    question=QUESTION_A,
    company="NVIDIA",
    reporting_period="fiscal 2026",
)
prompt_b = build_full_context_prompt(
    document_text=source_document,
    question=QUESTION_B,
    company="NVIDIA",
    reporting_period="fiscal 2026",
)

common_prefix_characters = 0
for first, second in zip(prompt_a, prompt_b, strict=False):
    if first != second:
        break
    common_prefix_characters += 1

common_prefix_tokens = estimate_tokens(prompt_a[:common_prefix_characters])
print(f"Prompt A estimate: {estimate_tokens(prompt_a):,} tokens")
print(f"Prompt B estimate: {estimate_tokens(prompt_b):,} tokens")
print(f"Reusable exact prefix: ~{common_prefix_tokens:,} tokens")
print("Question appears after document:", prompt_a.index("<question>") > prompt_a.index("</source_document>"))

> **Interpretation:** OpenAI currently requires a sufficiently long exact prefix for
> caching eligibility, while Ollama exposes different local runtime behavior. The
> portable engineering contract is the stable prefix; cache telemetry is an optional
> provider-specific observation.

## 4. Ask two questions over the same document

Only this cell crosses the model boundary. Everything before it is deterministic and
testable without a provider.

In [ ]:
OFFLINE_MODEL_NAME = "recorded-response-v2"
settings = Settings.from_environment()
live_mode = os.getenv("FINAI_LIVE_MODE", "1") == "1"
model = create_chat_model(settings) if live_mode else RecordedChatModel()

print("Execution mode:", "live" if live_mode else "offline fixture")
print("Provider configuration:", provider_summary(settings))

In [ ]:
def invoke_with_metrics(chat_model: Any, prompt: str) -> tuple[ModelRun, Any]:
    started = perf_counter()
    response = chat_model.invoke([
        ("system", SYSTEM_INSTRUCTIONS),
        ("human", prompt),
    ])
    latency_ms = (perf_counter() - started) * 1_000
    run = ModelRun(
        provider=settings.provider if live_mode else "offline",
        model=settings.chat_model if live_mode else OFFLINE_MODEL_NAME,
        text=str(response.content),
        latency_ms=latency_ms,
    )
    return run, response


run_a, response_a = invoke_with_metrics(model, prompt_a)
run_b, response_b = invoke_with_metrics(model, prompt_b)

print("QUESTION A\n", run_a.text)
print("\nQUESTION B\n", run_b.text)

### Visual 3 — Measure the two observed calls

The second call may benefit from reuse, but latency alone cannot prove a cache hit.
Inspect provider usage metadata when it is available.

In [ ]:
latencies_ms = pd.Series({"Question A": run_a.latency_ms, "Question B": run_b.latency_ms})
if live_mode:
    plotted_latencies = latencies_ms
    latency_unit = "milliseconds"
    latency_labels = [f"{value:,.0f} ms" for value in plotted_latencies.values]
    latency_title = "Observed model latency for a repeated document prefix"
else:
    plotted_latencies = latencies_ms * 1_000
    latency_unit = "microseconds · recorded fixture only"
    latency_labels = [f"{value:,.1f} µs" for value in plotted_latencies.values]
    latency_title = "Recorded fixture runtime — live mode shows provider latency"

fig, ax = plt.subplots(figsize=(8, 3.8))
bars = ax.bar(plotted_latencies.index, plotted_latencies.values, color=[NAVY, TEAL], width=0.55)
ax.bar_label(bars, labels=latency_labels, padding=4)
ax.set_title(latency_title)
ax.set_ylabel(f"Runtime ({latency_unit})")
ax.spines[["top", "right"]].set_visible(False)
ax.text(
    0.5,
    -0.22,
    "A faster second call is not proof of caching; use provider token telemetry.",
    transform=ax.transAxes,
    ha="center",
    color=SLATE,
)
plt.tight_layout()
plt.show()

print("Question A metadata:", getattr(response_a, "response_metadata", {}))

### Check grounding before trusting fluency

The visible checklist is intentionally simple. Later evaluation lessons will add a
gold question set and citation-level metrics. **Live grounding remains an observation**:
the deterministic offline response must pass, while a live provider may expose a
grounding REVIEW without invalidating the separate CAG/RAG routing boundary.

In [ ]:
grounding_result = evaluate_grounding(run_a.text)
for criterion, passed in grounding_result.checks.items():
    print(f"{'PASS' if passed else 'REVIEW':6} {criterion}")
print(f"Grounding score: {grounding_result.score}/{grounding_result.maximum}")

## Failure lab

We now place the relevant facts in the middle of a much longer **synthetic stress
document**. The added passages contain no financial claims. They exist only to test
the context policy reproducibly.

The failure is deterministic: if the full prompt exceeds the allocated budget, the
application must stop before silently truncating the filing or gambling on model
attention.

In [ ]:
SYNTHETIC_APPENDIX = (
    "TRAINING STRESS TEXT — no financial evidence. This neutral appendix exists "
    "only to increase context length and must not be used to answer the question."
)

filler_blocks = [
    f"[SYNTHETIC-{index:03d}] {SYNTHETIC_APPENDIX} " + ("context " * 140)
    for index in range(80)
]
midpoint = len(filler_blocks) // 2
stress_document = "\n\n".join(
    filler_blocks[:midpoint] + [source_document] + filler_blocks[midpoint:]
)

stress_document_tokens = estimate_tokens(stress_document)
stress_decision = decide_context_route(
    document_tokens=stress_document_tokens,
    system_prompt_tokens=components["Instructions"],
    question_tokens=components["Question"],
    budget=budget,
)
stress_fits = stress_decision.route == "cag"

evidence_start = stress_document.index("[F1]") / len(stress_document)
evidence_end = (stress_document.index("[F8]") + len(document_sections[-1]["text"])) / len(stress_document)

print(f"Stress document: ~{stress_document_tokens:,} tokens")
print(f"Evidence position: {evidence_start:.1%} to {evidence_end:.1%} of the document")
print("Reason:", stress_decision.reason)
print("Decision:", "CAG fits" if stress_fits else "RAG required")

### Visual 4 — The evidence is literally lost in the middle

This strip shows document position, not model quality. It makes the stress-test
construction auditable.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.barh([0], [1], color=LIGHT, height=0.34)
ax.barh(
    [0],
    [evidence_end - evidence_start],
    left=[evidence_start],
    color=TEAL,
    height=0.34,
)
ax.axvline(0.5, color=NAVY, linestyle="--", linewidth=1.5)
ax.text((evidence_start + evidence_end) / 2, 0, "NVIDIA evidence", ha="center", va="center", color="white")
ax.text(0.5, 0.32, "middle of context", ha="center", color=NAVY)
ax.set_xlim(0, 1)
ax.set_xticks(np.linspace(0, 1, 5), labels=["0%", "25%", "50%", "75%", "100%"])
ax.set_yticks([])
ax.set_xlabel("Relative document position")
ax.set_title("Evidence location inside the synthetic long document")
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()
plt.show()

### Visual 5 — Make the CAG/RAG boundary explicit

The decision is an application policy. It depends on the selected model window, answer
reservation and prompt overhead—not on a universal document-size rule.

In [ ]:
stress_levels = np.arange(0, 81, 10)
document_sizes = []
decisions = []
for count in stress_levels:
    candidate = "\n\n".join(
        filler_blocks[: count // 2] + [source_document] + filler_blocks[count // 2 : count]
    )
    size = estimate_tokens(candidate)
    document_sizes.append(size)
    decisions.append(
        should_use_full_context(
            document_tokens=size,
            system_prompt_tokens=components["Instructions"],
            question_tokens=components["Question"],
            budget=budget,
        )
    )

fig, ax = plt.subplots()
ax.plot(stress_levels, document_sizes, marker="o", color=NAVY, linewidth=2.5)
ax.axhline(
    budget.available_input_tokens - components["Instructions"] - components["Question"],
    color=RED,
    linestyle="--",
    linewidth=2,
    label="Maximum document allocation",
)
first_rejected = next(index for index, fits in enumerate(decisions) if not fits)
ax.axvspan(stress_levels[first_rejected], stress_levels[-1], color=RED, alpha=0.08, label="RAG region")
ax.set_title("When the complete document stops fitting")
ax.set_xlabel("Synthetic appendix blocks added")
ax.set_ylabel("Estimated document tokens")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print("Decision: RAG required" if not stress_fits else "Decision: CAG fits")

## Verification

These checks verify the engineering boundary. They do not claim that an answer is
financially correct merely because the prompt fits.

In [ ]:
checks = {
    "real document pack fits": base_fits,
    "stress document is rejected": not stress_fits,
    "base route is explicit": base_decision.route == "cag",
    "stress route explains the budget failure": (
        stress_decision.route == "rag" and "exceeds" in stress_decision.reason
    ),
    "document precedes question": prompt_a.index("</source_document>") < prompt_a.index("<question>"),
    "model returned an answer": bool(run_a.text.strip()),
    "visual decision boundary exists": any(decisions) and not all(decisions),
}

for criterion, passed in checks.items():
    print(f"{'PASS' if passed else 'REVIEW':6} {criterion}")

if live_mode:
    print("Live grounding observation:", "PASS" if grounding_result.passed else "REVIEW")
if not live_mode:
    assert grounding_result.passed, "The recorded answer must pass the grounding contract."

assert all(checks.values()), "Review the visible CAG checks before continuing."
print("PASS — CAG boundary verified")

## Challenge

You receive a second document containing 5,500 estimated tokens. Your model has an
8,192-token window, the system instructions and question use 650 tokens, and you reserve
1,500 tokens for the answer.

1. Use `ContextBudget` and `decide_context_route` to decide whether CAG fits.
2. Change only the reserved output to 2,200 tokens and recompute.
3. Write a two-sentence engineering decision record: CAG or RAG, and why.

**Expected reasoning:** the second decision must change because output capacity is part
of the context contract.

## Troubleshooting

| Symptom | Likely cause | Action |
|---|---|---|
| `Ollama support is not installed` | optional AI dependencies missing | run the repository setup command with the `ai` extra |
| connection refused on port 11434 | Ollama is not running | start Ollama and confirm the configured model exists |
| OpenAI authentication error | missing or invalid key | set `OPENAI_API_KEY`; never paste it into the notebook |
| context error or truncation | selected model window is smaller | update the explicit budget and rerun the decision cell |
| second call is not faster | no cache hit, local variance or cold start | inspect provider usage metadata; do not infer caching from latency alone |
| live answer fails grounding | output omitted citations or limitation | strengthen the question contract, then rerun only the call and checks |

## Knowledge check

1. Why is Cache different from Grounding?
2. Which ContextDecision field explains why the route changed?
3. Does Memory replace RAG for a large audited document corpus?

**Answers:** cache may reuse bytes but does not prove evidence support; `reason` records
the budget comparison; memory retains interaction state while RAG selects source evidence.

## Capstone integration

The Financial Analyst Copilot now has a second analysis path:

```text
small bounded document → full-context prompt → grounded answer
large or growing corpus → stop at the budget gate → hand off to RAG
```

Save the decision variables with each run: document tokens, prompt overhead, output
reservation, selected model window and the resulting route. This becomes observable
application state in the evaluation lesson.

## Recap

- Context engineering decides what the model can see.
- CAG is effective when the complete bounded source fits and related requests reuse a stable prefix.
- Prompt caching is a provider optimization, not a grounding mechanism.
- A deterministic budget gate prevents silent truncation.
- The long-document failure gives Lesson 4 a concrete job: retrieve only the evidence needed.